# Portfolio default simulation

From **name-level PDs** to a **systematic factor**, a **copula**, **correlated default indicators**, and a **portfolio loss distribution** — with a Gaussian vs Student-$t$ tail comparison.

## Concept

In a **one-factor** portfolio model, a single market draw $Z$ shifts all obligors’ **conditional** default probabilities. Names load on that factor through **asset correlation** $\rho$ and a **copula** (Gaussian or Student-$t$) that maps factor realizations to uniform margins.

The simulation loop is: draw $Z$; for each name compute $P(\text{default} \mid Z)$ via `conditional_default_prob` with threshold $c_i = \Phi^{-1}(\mathrm{PD}_i)$; Bernoulli-draw defaults; aggregate **loss** $= \sum \mathbf{1}_{\text{default},i} \cdot N_i \cdot \mathrm{LGD}_i$. Repeating yields a **loss distribution**; we summarize **mean**, **VaR(99%)**, and **ES(99%)**.

Student-$t$ copulas exhibit **tail dependence**: conditional default probabilities react differently in joint stress than under a Gaussian copula, which often shows **fatter portfolio loss tails**.

## API walkthrough

Key pieces from `finstack.correlation`:

- **`CopulaSpec` / `Copula`** — `.build()` gives a copula; `conditional_default_prob(threshold, [Z], rho)` returns conditional PD given factor $Z$ and asset correlation $\rho$.
- **`CorrelatedBernoulli`** and **`joint_probabilities`** — two-name joint default mass for a chosen correlation.
- **`correlation_bounds`**, **`validate_correlation_matrix`**, **`cholesky_decompose`** — Fréchet–Hoeffding feasibility, PSD checks, and Cholesky factors for flat row-major correlation matrices.

Use **`standard_normal_inv_cdf`** from `finstack.core.math.special_functions` for thresholds $c_i = \Phi^{-1}(\mathrm{PD}_i)$.

In [ ]:
from finstack.correlation import (
    CopulaSpec,
    CorrelatedBernoulli,
    RecoverySpec,
    correlation_bounds,
    joint_probabilities,
    validate_correlation_matrix,
    cholesky_decompose,
)
from finstack.core.math.special_functions import norm_cdf, standard_normal_inv_cdf
import math
import random

print(
    "Imports OK: finstack.correlation (CopulaSpec, bounds, Cholesky, Bernoulli) + inv-cdf"
)

Fréchet–Hoeffding **correlation bounds** for two Bernoulli marginals, a tiny **2-name** joint view, and a **3×3** correlation matrix check plus its Cholesky factor.

In [ ]:
lo, hi = correlation_bounds(0.02, 0.05)
print(f"Feasible correlation range for PDs (0.02, 0.05): [{lo:.4f}, {hi:.4f}]")

p11, p10, p01, p00 = joint_probabilities(0.02, 0.05, 0.15)
print(f"Joint probs (p11,p10,p01,p00) at rho=0.15: ({p11:.6f}, {p10:.6f}, {p01:.6f}, {p00:.6f})")
print(f"Sum check: {p11 + p10 + p01 + p00:.10f}")

cb = CorrelatedBernoulli(0.02, 0.05, 0.15)
print(f"CorrelatedBernoulli: {cb}")
print(f"P(both default): {cb.joint_p11:.6f}")

flat_3 = [1.0, 0.2, 0.1, 0.2, 1.0, 0.15, 0.1, 0.15, 1.0]
validate_correlation_matrix(flat_3, 3)
print("validate_correlation_matrix: 3x3 PSD example accepted")
L = cholesky_decompose(flat_3, 3)
print(f"Cholesky (flat, len={len(L)}): first 3 entries = {L[:3]}")

**Gaussian copula** — build once; thresholds $c_i = \Phi^{-1}(\mathrm{PD}_i)$; `tail_dependence` is zero. **Student-$t$** (e.g. 5 df) has **non-zero** tail dependence for the same $\rho$.

In [ ]:
gauss = CopulaSpec.gaussian().build()
t_cop = CopulaSpec.student_t(5.0).build()
rho_demo = 0.20
c_demo = standard_normal_inv_cdf(0.03)
print(f"Gaussian model: {gauss.model_name}, tail_dep(0.5)={gauss.tail_dependence(0.5):.6f}")
print(f"Student-t model: {t_cop.model_name}, tail_dep(0.5)={t_cop.tail_dependence(0.5):.6f}")
print(
    f"Same Z=-2.0: Gauss cond PD={gauss.conditional_default_prob(c_demo, [-2.0], rho_demo):.6f}, "
    f"t cond PD={t_cop.conditional_default_prob(c_demo, [-2.0], rho_demo):.6f}"
)

_ = RecoverySpec.constant(0.40)
print("RecoverySpec.constant(0.40) available for follow-on notebooks (no effect here)")

## Examples

**10-name** portfolio: individual PDs, **LGD 60%**, equal notionals **10M**, single-factor asset correlation **0.20**. **10,000** paths, `random.seed(42)`. Compare **Gaussian** vs **Student-$t$ (5 df)** loss distributions via **mean**, **VaR(99%)**, and **ES(99%)** (tail mean above VaR).

In [ ]:
def var_and_es(losses: list[float], alpha: float = 0.99) -> tuple[float, float]:
    xs = sorted(losses)
    n = len(xs)
    idx = min(n - 1, max(0, math.ceil(alpha * n) - 1))
    var_v = xs[idx]
    tail = [x for x in xs if x >= var_v]
    es_v = sum(tail) / len(tail) if tail else var_v
    return var_v, es_v


def simulate_losses(copula, n_sims: int, rng: random.Random) -> list[float]:
    out: list[float] = []
    for _ in range(n_sims):
        z = rng.gauss(0.0, 1.0)
        portfolio_loss = 0.0
        for i in range(n_names):
            c_i = standard_normal_inv_cdf(pds[i])
            cond_pd = copula.conditional_default_prob(c_i, [z], rho)
            if rng.random() < cond_pd:
                portfolio_loss += notionals[i] * lgds[i]
        out.append(portfolio_loss)
    return out


n_names = 10
pds = [0.02, 0.03, 0.05, 0.02, 0.04, 0.03, 0.06, 0.01, 0.03, 0.05]
lgds = [0.60] * n_names
notionals = [10_000_000.0] * n_names
rho = 0.20
n_sims = 10_000

rng_g = random.Random(42)
rng_t = random.Random(42)
losses_g = simulate_losses(gauss, n_sims, rng_g)
losses_t = simulate_losses(t_cop, n_sims, rng_t)

mean_g = sum(losses_g) / n_sims
mean_t = sum(losses_t) / n_sims
var_g, es_g = var_and_es(losses_g)
var_t, es_t = var_and_es(losses_t)

print(f"Simulations: {n_sims}, seed=42 (separate RNGs, same draw sequence)")
print(f"Gaussian  — mean loss: {mean_g:,.0f}, VaR99: {var_g:,.0f}, ES99: {es_g:,.0f}")
print(f"Student-t — mean loss: {mean_t:,.0f}, VaR99: {var_t:,.0f}, ES99: {es_t:,.0f}")
print(
    f"Tail uplift (t vs Gauss): VaR99 {((var_t / var_g) - 1.0) * 100:+.2f}%, "
    f"ES99 {((es_t / es_g) - 1.0) * 100:+.2f}% (ratio not defined if VaR=0)"
)
print(f"Check unconditional PD of name with PD=0.01: Phi(c)={norm_cdf(standard_normal_inv_cdf(0.01)):.6f}")

## Takeaways

- **Thresholds** passed to the copula are **Gaussian** quantiles $c_i = \Phi^{-1}(\mathrm{PD}_i)$, not the raw PD.
- **One factor** $Z$ drives all names; **asset correlation** $\rho$ controls how sharply conditional PDs move with $Z$.
- **Gaussian** vs **Student-$t$** copulas can match similar **unconditional** behavior but differ in **joint stress**; tail risk metrics (**VaR / ES**) often **rise** under $t$ for the same simulation design.
- **Fréchet–Hoeffding** bounds remind you that not every pairwise correlation is **feasible** for given PDs; **`validate_correlation_matrix`** and **`cholesky_decompose`** support multi-name extensions.